In [3]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [12]:
from selenium import webdriver
options = webdriver.ChromeOptions()
options.add_argument('--ignore-certificate-errors')
driver = webdriver.Chrome(options=options)
driver.get('https://sh.lianjia.com/ershoufang/')#'您的网址'
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

In [13]:
import urllib.request as req
from bs4 import BeautifulSoup
import csv
import time
import random
import re
import requests

In [14]:
import os
 
os.chdir('F:\pythonProject') #更改当前目录，所有的信息csv将会保存在这里。不可保存在磁盘根目录下

In [15]:
#由于当前lianjia需要登陆才能翻页查阅，本更新版本利用selenium模拟登陆，需要先install selenium和webdriver
# 初始化浏览器 (这里以 Chrome 为例)
driver = webdriver.Chrome()  # 或者指定路径 webdriver.Chrome(executable_path='path_to_chromedriver')
# 打开登录页面
driver.get("https://clogin.lianjia.com/login?service=https%3A%2F%2Fwww.lianjia.com%2Fuser%2Fchecklogin%3Fredirect%3Dhttps%253A%252F%252Fsh.lianjia.com%252Fershoufang%252Fbaoshan%252F")
#driver.implicitly_wait(100)
#可以看到Chrome弹出，保持该窗口开启

In [16]:
# 等待页面加载并找到用户名和密码输入框
username_field = driver.find_element(By.ID, "username")
username_field.send_keys("username")  # 替换为实际用户名
password_field = driver.find_element(By.ID, "password")  # 根据页面结构修改 ID 或其他定位方式
password_field.send_keys("password") # 替换为实际密码

In [17]:
login_button = driver.find_element(By.XPATH, "//button[@type='submit']")  # 根据页面结构修改 ID 或其他定位方式
login_button.click()
#可能会需要动态手机验证 只能手动，验证后需要保持该chrome窗口打开，如长时间不操作可能会断开，需要从初始化浏览器的步骤重新连接

In [18]:
#获取lianjia上的上海全域分区，可跳过
soup = BeautifulSoup(driver.page_source, 'lxml')
a = soup.find("div", {"data-role": "ershoufang"}).findAll("a")
areaList = []
for n in a:
    areaName = n.get('title')
    if areaName == None:
        continue
    else:
        areaLink = n.get('href')
        found = re.search('/ershoufang/(.+?)/', areaLink).group(1)
        print(found)
        areaList.append(found)
print(areaList)

pudong
minhang
baoshan
xuhui
putuo
yangpu
changning
songjiang
jiading
huangpu
jingan
hongkou
qingpu
fengxian
jinshan
chongming
shanghaizhoubian
['pudong', 'minhang', 'baoshan', 'xuhui', 'putuo', 'yangpu', 'changning', 'songjiang', 'jiading', 'huangpu', 'jingan', 'hongkou', 'qingpu', 'fengxian', 'jinshan', 'chongming', 'shanghaizhoubian']


In [19]:
#本更新版本还是使用beautifulsoup演示，但所有定位操作同样可以利用selenium完成
def get_inf(area, page=1):
    # 获得总页数total
    total = 100 #shanghai 每个区当前都超过一百页，最多显示一百页
    # 从1到最后页翻页
    host = "https://sh.lianjia.com/ershoufang/"
    for i in range(page, total + 1):
        print("page" + str(i))
        if i == 1:  # 第一页网址
            pagehtml = host + area+ "/"        
        else:  # 其他页网址
            pagehtml = host + area + "/pg" + str(i) + "/"
        driver.get(pagehtml)
        pageSoup = BeautifulSoup(driver.page_source, "lxml")
        # 获得当前页面的所有房屋列表
        url_list = pageSoup.find("ul", {"class": "sellListContent"}).findAll("div", {"class": "title"})
        #print(url_list)
        print(len(url_list))
        # 历遍房屋列表
        for l in url_list:
            de_url = l.find('a')['href']
            print(de_url)
            with open(area + '.csv', 'a', newline='') as f:  # 创建csv
                writer = csv.writer(f)
                try:
                    driver.get(de_url)  # 访问房屋网址,timeout设置超时的时间
                    soup = BeautifulSoup(driver.page_source, "lxml")  # 解析网址
                    name = soup.find("h1", {"class": "main"}).get_text()  # 获得房屋名称
                    price = soup.find("span", {"class": "unitPriceValue"}).get_text()  # 获得房屋均价
                    communityName = soup.find("div", {"class": "communityName"}).find("a", {"class": "info"}).get_text()  # 获得小区名称
                    build_year = soup.find("div", {"class": "area"}).find("div", {"class": "subInfo"}).get_text()  # 获得年份
                    datas = soup.find("div", {"class": "base"}).findAll("li")  # 获得其他信息
                    lst = []
                    for data in datas:
                        lst.append(data.get_text())
                    # 写入文件
                    # '房屋名称', '房屋均价', '小区名称', '年份', '房屋户型', '所在楼层', '建筑面积', '户型结构', '套内面积', '建筑类型', '房屋朝向', '建筑结构', '装修情况','梯户比例', '装备电梯'
                    writer.writerow([name, price, communityName, build_year, lst[0], lst[1], lst[2], lst[3], lst[4], lst[5], lst[6], lst[7], lst[8], lst[9], lst[10], i])
                    # 模拟人访问在网页停留时间
                    time.sleep(16 + random.randint(0, 9))
                except Exception as e:
                    print(str(e))

In [20]:
#调整目标area，可不跑全城
#areaList = ['pudong', 'minhang', 'baoshan', 'xuhui', 'putuo', 'yangpu', 'changning', 'songjiang', 'jiading', 'huangpu', 'jingan', 'hongkou', 'qingpu', 'fengxian', 'jinshan', 'chongming', 'shanghaizhoubian']
areaList = ['putuo']


In [21]:
# 历遍每一个区域
for j in areaList:
    area = j
    print(area)
    get_inf(area)
print("finished")

#为保持chromedriver持续连接，需要保持计算机不要睡眠

putuo
page1
30
https://sh.lianjia.com/ershoufang/107110465485.html
https://sh.lianjia.com/ershoufang/107111280240.html
https://sh.lianjia.com/ershoufang/107111036463.html
https://sh.lianjia.com/ershoufang/107111257838.html
https://sh.lianjia.com/ershoufang/107110309253.html
https://sh.lianjia.com/ershoufang/107107432212.html
https://sh.lianjia.com/ershoufang/107111280672.html
https://sh.lianjia.com/ershoufang/107110216310.html
https://sh.lianjia.com/ershoufang/107111105009.html
https://sh.lianjia.com/ershoufang/107108693720.html
https://sh.lianjia.com/ershoufang/107111017359.html


KeyboardInterrupt: 

In [ ]:
#关闭窗口
driver.quit()